# Moirai2 + GNN — VN30 Volatility (Google Colab GPU)

**Hướng dẫn:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Chạy **Cell 1** (config) → **Cell 2** (clone) → **Cell 3** (install → tự restart)
3. Sau restart: chạy **Cell 2** (git pull) rồi chạy **từ Cell 4 trở đi** — Cell 3 tự skip

> Dùng Run All: sau restart, Run All lại — Cell 3 skip tự động, Cell 2 sẽ git pull.

In [ ]:
# ── Cấu hình — không cần sửa gì ─────────────────────────────────────────────
REPO_URL = 'https://github.com/ntquy9901/vn30-volatility-gnn.git'
# ─────────────────────────────────────────────────────────────────────────────

import torch
print(f'torch : {torch.__version__}')
print(f'CUDA  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────────
import os, subprocess, sys

WORKDIR = '/content/moirai'

if not os.path.exists(WORKDIR):
    result = subprocess.run(['git', 'clone', REPO_URL, WORKDIR],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print('CLONE FAILED:', result.stderr)
        sys.exit(1)
    print('Cloned OK')
else:
    subprocess.run(['git', '-C', WORKDIR, 'pull'], check=True)
    print('Pulled latest')

os.chdir(WORKDIR)
print(f'Working dir: {os.getcwd()}')
print('Files:', sorted(os.listdir('.')))
sys.path.insert(0, WORKDIR)
os.environ['HF_HOME'] = '/content/hf_cache'
print('Path ready.')
import torchvision  # must load before torch_geometric to avoid circular import

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
# Chạy cell này một lần — sau khi install xong sẽ tự restart kernel.
# Sau restart: bỏ qua cell này, chạy từ cell tiếp theo.
import subprocess, sys, os

_MARKER = '/content/.deps_installed'
if not os.path.exists(_MARKER):
    print('Installing packages (sẽ restart kernel sau khi xong)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'numpy==1.26.4',          # pin để tránh binary incompatibility
        'torch-geometric',
        'uni2ts',
        'gluonts',
        'arch',
    ])
    open(_MARKER, 'w').close()
    print('Done. Restarting kernel...')
    os.kill(os.getpid(), 9)       # hard restart — re-run từ cell kế tiếp
else:
    print('Packages already installed, skipping.')


In [ ]:
# ── 3. Load config ────────────────────────────────────────────────────────────
import yaml
import pandas as pd

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

# Data đã có trong repo, path relative là đúng
print(f"prices_dir : {cfg['data']['prices_dir']}")
print(f"train_end  : {cfg['data']['train_end']}")
print(f"test_start : {cfg['data']['test_start']}")
print(f"stride     : {cfg['model']['stride']} days")
print(f"epochs     : {cfg['training']['epochs']}")

In [ ]:
# ── 4. Kiểm tra data ──────────────────────────────────────────────────────────
from src.volatility_labels import load_close_prices, compute_log_returns, compute_rv
from gnn.build_graph import VN30_TICKERS

close = load_close_prices(cfg['data']['prices_dir'], tickers=VN30_TICKERS + ['VNINDEX'])
print(f'Prices: {close.shape}  {close.index[0].date()} → {close.index[-1].date()}')
print(f'Tickers loaded: {close.notna().any().sum()}/31')

In [ ]:
# ── 5. Train GNN (walk-forward) ───────────────────────────────────────────────
import time
from gnn.train import train_walkforward

RESULTS = '/content/results'
os.makedirs(RESULTS, exist_ok=True)

print('Starting GNN walk-forward training...')
t0 = time.time()
metrics_df = train_walkforward(cfg, results_dir=RESULTS)
elapsed = time.time() - t0

print(f'\nDone: {len(metrics_df)} windows in {elapsed/60:.1f} min')
print(metrics_df.tail(5).to_string(index=False))

In [ ]:
# ── 6. Train MLP+Moirai2 (ablation baseline, no graph) ───────────────────────
from baselines.mlp_baseline import train_mlp_walkforward

print('Training MLP+Moirai2...')
t0 = time.time()
train_mlp_walkforward(cfg, results_dir=RESULTS)
print(f'Done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── 7. GNN inference trên test set 2026 ──────────────────────────────────────
import torch
import numpy as np
from src.embed_extractor import Moirai2Embedder
from gnn.model import VolatilityGNN
from gnn.build_graph import build_graph
from gnn.train import extract_embeddings

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log_ret   = compute_log_returns(close)
rv_all    = compute_rv(close[VN30_TICKERS], h=cfg['model']['horizon'])
TEST_START = pd.Timestamp(cfg['data']['test_start'])
test_dates = rv_all.index[(rv_all.index >= TEST_START) & (~rv_all.isna().all(axis=1))]

gnn_model = VolatilityGNN(
    in_dim=Moirai2Embedder.D_MODEL,
    hidden=cfg['model']['gnn_hidden'],
    mlp_hidden=cfg['model']['mlp_hidden'],
    dropout=cfg['model']['dropout'],
).to(device)
gnn_model.load_state_dict(torch.load(f'{RESULTS}/best_gnn.pt', map_location=device))
gnn_model.eval()

embedder = Moirai2Embedder(
    size='small',
    context_length=cfg['model']['context_length'],
    patch_size=cfg['model'].get('patch_size', 32),
)

print(f'Inference on {len(test_dates)} test dates ({device})...')
gnn_preds = {t: {} for t in VN30_TICKERS}
with torch.no_grad():
    for i, date in enumerate(test_dates):
        g     = build_graph(log_ret, end_date=date,
                            corr_window=cfg['model']['corr_window'],
                            corr_threshold=cfg['model']['corr_threshold'])
        feats = extract_embeddings(embedder, log_ret, date, cfg['model']['context_length']).to(device)
        pred  = gnn_model(feats, g.edge_index.to(device)).cpu().numpy().ravel()
        for j, tk in enumerate(VN30_TICKERS):
            gnn_preds[tk][date] = max(float(pred[j+1]), 0.0)
        if (i+1) % 20 == 0: print(f'  {i+1}/{len(test_dates)}')

gnn_pred_df = pd.DataFrame({t: pd.Series(gnn_preds[t]) for t in VN30_TICKERS})
print(f'GNN predictions: {gnn_pred_df.shape}')

In [ ]:
# ── 8. MLP+Moirai2 inference ──────────────────────────────────────────────────
from baselines.mlp_baseline import run_mlp_inference
mlp_pred_df = pd.DataFrame(
    run_mlp_inference(cfg, checkpoint_path=f'{RESULTS}/best_mlp.pt', results_dir=RESULTS)
)
print(f'MLP predictions: {mlp_pred_df.shape}')

In [ ]:
# ── 9. Statistical baselines ──────────────────────────────────────────────────
from baselines.garch_baseline import run_garch_baseline
from baselines.har_rv_baseline import run_har_baseline
from baselines.lstm_baseline import run_lstm_baseline

kwargs = dict(prices_dir=cfg['data']['prices_dir'],
              train_end=cfg['data']['train_end'],
              test_start=cfg['data']['test_start'],
              horizon=cfg['model']['horizon'])

print('GARCH...');  garch_df = pd.DataFrame(run_garch_baseline(**kwargs))
print('HAR-RV...');  har_df  = pd.DataFrame(run_har_baseline(**kwargs))
print('LSTM...');   lstm_df  = pd.DataFrame(run_lstm_baseline(**kwargs))
print(f'Shapes — GARCH:{garch_df.shape} HAR:{har_df.shape} LSTM:{lstm_df.shape}')

In [ ]:
# ── 10. Bảng kết quả ─────────────────────────────────────────────────────────
from evaluation.metrics import compare_models

def pool(pred_df, true_df):
    common = pred_df.index.intersection(true_df.index)
    yt = true_df.loc[common].values.ravel()
    yp = pred_df.loc[common].values.ravel()
    v  = ~(np.isnan(yt) | np.isnan(yp))
    return yt[v], yp[v]

yt, yp_gnn   = pool(gnn_pred_df,  rv_all)
_,  yp_mlp   = pool(mlp_pred_df,  rv_all)
_,  yp_garch = pool(garch_df,     rv_all)
_,  yp_har   = pool(har_df,       rv_all)
_,  yp_lstm  = pool(lstm_df,      rv_all)

n = len(yt)
def align(a): return a[:n] if len(a) >= n else np.concatenate([a, np.full(n-len(a), np.nan)])

results = compare_models(
    yt,
    {'GNN (ours)': yp_gnn, 'MLP+Moirai2': align(yp_mlp),
     'GARCH(1,1)': align(yp_garch), 'HAR-RV': align(yp_har), 'LSTM': align(yp_lstm)},
    dm_reference='GNN (ours)',
    dm_loss=cfg['evaluation']['dm_loss'],
)

print('\n=== Kết quả (test 2026, pooled 30 cổ phiếu) ===\n')
print(results[['MAE','RMSE','R2','QLIKE','Pearson_r','DM_stat','DM_pval']].round(4).to_string())

results.to_csv(f'{RESULTS}/model_comparison.csv')

# Graph contribution
d = (results.loc['MLP+Moirai2','MAE'] - results.loc['GNN (ours)','MAE']) / results.loc['MLP+Moirai2','MAE'] * 100
print(f'\nGraph contribution (MAE): {d:+.1f}%  => {"Graph HELPS" if d>0 else "Graph does NOT help"}')

In [ ]:
# ── 11. Visualizations ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from evaluation.metrics import r2_score as ev_r2

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'QLIKE']):
    vals   = results[metric]
    colors = ['#1565C0' if i=='GNN (ours)' else '#42A5F5' if i=='MLP+Moirai2' else '#90CAF9'
              for i in vals.index]
    bars = ax.bar(vals.index, vals, color=colors)
    ax.set_title(metric); ax.tick_params(axis='x', rotation=20)
    ax.bar_label(bars, fmt='%.4f', fontsize=8)
plt.suptitle('Model Comparison — 2026 Test Set', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Per-stock R² heatmap
test_idx = rv_all.index[rv_all.index >= TEST_START]
r2_tbl   = pd.DataFrame(index=VN30_TICKERS, columns=['GNN','MLP+M2','HAR','GARCH','LSTM'])
for tk in VN30_TICKERS:
    rv_t = rv_all[tk].reindex(test_idx).dropna()
    for col, df in [('GNN',gnn_pred_df),('MLP+M2',mlp_pred_df),
                    ('HAR',har_df),('GARCH',garch_df),('LSTM',lstm_df)]:
        if tk not in df.columns: continue
        sh = rv_t.index.intersection(df[tk].dropna().index)
        if len(sh) > 5:
            r2_tbl.loc[tk, col] = ev_r2(rv_t.loc[sh].values, df[tk].loc[sh].values)
r2_tbl = r2_tbl.astype(float)
plt.figure(figsize=(8, 10))
sns.heatmap(r2_tbl, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-0.5, vmax=0.8, linewidths=0.5)
plt.title('Per-stock R² — 2026 Test Set')
plt.tight_layout()
plt.savefig(f'{RESULTS}/r2_heatmap.png', dpi=150)
plt.show()
print('\nMean R²:', r2_tbl.mean().round(3).to_string())

In [ ]:
# ── 12. Download kết quả về máy ───────────────────────────────────────────────
import shutil
from google.colab import files

# Zip toàn bộ results
shutil.make_archive('/content/results', 'zip', RESULTS)
files.download('/content/results.zip')
print('results.zip đã download — giải nén vào moirai/results/ trên máy.')